# 03 · Species in Bounding Box (Plantae + Invasive Status)

**Task**: Query GBIF occurrence API for a geographic bounding box, retrieve all **Plantae** species,
and classify each as **invasive** or **non-invasive** based on `establishmentMeans` and `degreeOfEstablishment`.

**Target area**: Galicia coast (NW Spain) – occurrence data from bbox only
- West: -8.80, South: 43.15, East: -8.10, North: 43.58
- **Invasive species list**: from all Spain (GBIF facet with geometry returns 0 for small bbox)
- Kingdom: Plantae (Cortaderia selloana is a grass – Poaceae)
- Year filter: 2024 (configurable)

**Output**:
- List of species with occurrence counts, invasive status, and establishment info
- Highlight *Cortaderia selloana* (pampas grass – invasive)
- Save to `data/species_in_bbox.csv` and `data/species_in_bbox.json`

In [1]:
%pip install -q requests pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [56]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from tqdm.notebook import tqdm
from IPython.display import display

In [57]:
# ─── Configuration ────────────────────────────────────────────────────────────

# Bounding box: Galicia coast (NW Spain)
# Cortaderia selloana is Plantae (Poaceae – grasses)
BBOX = {
    "west":  -8.80,
    "south": 43.15,
    "east":  -8.10,
    "north": 43.58,
}

# WKT polygon (lon lat order, closed loop)
WKT_POLYGON = (
    f"POLYGON(({BBOX['west']} {BBOX['south']}, "
    f"{BBOX['east']} {BBOX['south']}, "
    f"{BBOX['east']} {BBOX['north']}, "
    f"{BBOX['west']} {BBOX['north']}, "
    f"{BBOX['west']} {BBOX['south']}))"
)

KINGDOM_KEY = 6   # Plantae
COUNTRY = "ES"    # invasive species list from all Spain (facet); records from Galicia bbox only
YEAR = 2024       # filter by year (None = all years)

# Known invasive species (Spain catalogue, GRIIS) – used when GBIF record lacks establishmentMeans
KNOWN_INVASIVE_SPECIES = {
    "cortaderia selloana",
    "ailanthus altissima",
    "opuntia ficus-indica",
    "eucalyptus globulus",
    "acacia dealbata",
    "carpobrotus edulis",
}
GBIF_PAGE_SIZE = 300
REQUEST_DELAY = 0.25
TIMEOUT = 30
MAX_RECORDS = 10_000   # GBIF API hard limit per query
MAX_RECORDS_TO_FETCH = 10_000   # limit downloads (Spain has many records); set to None to fetch all

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = DATA_DIR / "species_in_bbox.csv"
OUTPUT_JSON = DATA_DIR / "species_in_bbox.json"

TARGET_SPECIES = "Cortaderia selloana"

print(f"Bounding box: W={BBOX['west']} S={BBOX['south']} E={BBOX['east']} N={BBOX['north']}")
print(f"Kingdom: Plantae (key={KINGDOM_KEY})")
print(f"Year: {YEAR if YEAR else 'all'}")
print(f"Target species to highlight: {TARGET_SPECIES}")

Bounding box: W=-8.8 S=43.15 E=-8.1 N=43.58
Kingdom: Plantae (key=6)
Year: 2024
Target species to highlight: Cortaderia selloana


## 1 · Fetch occurrences from GBIF

We page through the occurrence search API with:
- `geometry` – WKT polygon for the bounding box
- `kingdomKey=6` – Plantae only
- `hasCoordinate=true` – records with coordinates

**Note**: GBIF occurrence records often have sparse `establishmentMeans`/`degreeOfEstablishment` coverage.
We include all species but mark establishment status per record. Species with at least one invasive
record are flagged as invasive.

In [58]:
def is_invasive_record(occ: dict) -> tuple[bool, str]:
    """
    Classify a single occurrence as invasive based on Darwin Core fields.
    Returns (is_invasive, establishment_detail).
    """
    doe = (occ.get("degreeOfEstablishment") or "").strip().lower()
    em  = (occ.get("establishmentMeans") or "").strip().upper()
    
    if doe == "invasive":
        return True, f"degreeOfEstablishment={doe}"
    if em in ("INVASIVE", "INTRODUCED", "NATURALISED", "NATURALIZED"):
        return True, f"establishmentMeans={em}"
    
    if em or doe:
        return False, f"establishmentMeans={em or '-'} degreeOfEstablishment={doe or '-'}"
    return False, "unknown"


def fetch_occurrences_in_bbox(max_records: int | None = None) -> list[dict]:
    """
    Page through GBIF occurrence search and collect records in the bbox.
    max_records: stop after this many records (None = fetch all, up to MAX_RECORDS).
    """
    limit = min(max_records, MAX_RECORDS) if max_records else MAX_RECORDS
    all_records = []
    offset = 0

    with tqdm(total=limit, desc="Fetching occurrences", unit="rec") as pbar:
        while True:
            params = {
                "geometry": WKT_POLYGON,
                "kingdomKey": KINGDOM_KEY,
                "hasCoordinate": "true",
                "limit": GBIF_PAGE_SIZE,
                "offset": offset,
            }
            if YEAR is not None:
                params["year"] = YEAR
            try:
                resp = requests.get(
                    "https://api.gbif.org/v1/occurrence/search",
                    params=params,
                    timeout=TIMEOUT,
                )
                resp.raise_for_status()
            except requests.RequestException as e:
                print(f"\n[ERROR] API request failed at offset={offset}: {e}")
                break

            data = resp.json()
            results = data.get("results", [])

            if not results:
                break

            n_before = len(all_records)
            for occ in results:
                if len(all_records) >= limit:
                    break
                sci_name = occ.get("scientificName") or occ.get("species") or ""
                if not sci_name and occ.get("genus"):
                    sci_name = occ.get("genus", "")
                if not sci_name:
                    continue

                inv, detail = is_invasive_record(occ)
                all_records.append({
                    "speciesKey": occ.get("speciesKey") or occ.get("taxonKey"),
                    "scientificName": sci_name.strip(),
                    "genus": occ.get("genus") or "",
                    "species": occ.get("species") or "",
                    "establishmentMeans": occ.get("establishmentMeans") or "",
                    "degreeOfEstablishment": occ.get("degreeOfEstablishment") or "",
                    "is_invasive": inv,
                    "establishment_detail": detail,
                })

            pbar.update(len(all_records) - n_before)

            if data.get("endOfRecords", False) or len(all_records) >= limit:
                break

            offset += GBIF_PAGE_SIZE
            time.sleep(REQUEST_DELAY)

    return all_records

In [61]:
# ─── Fetch invasive species from GBIF facets (all Spain – geometry+facet returns 0 for small bbox) ──
# degreeOfEstablishment=invasive + establishmentMeans=introduced
def _fetch_invasive_facet(filter_param: str, filter_value: str) -> set:
    params = {
        "country": COUNTRY,
        "kingdomKey": KINGDOM_KEY,
        "hasCoordinate": "true",
        filter_param: filter_value,
        "facet": "speciesKey",
        "facetLimit": 2000,
        "limit": 0,
    }
    if YEAR is not None:
        params["year"] = YEAR
    resp = requests.get("https://api.gbif.org/v1/occurrence/search", params=params, timeout=TIMEOUT)
    resp.raise_for_status()
    keys = set()
    for f in resp.json().get("facets", []):
        if str(f.get("field", "")).lower() in ("specieskey", "species_key"):
            for c in f.get("counts", []):
                try:
                    keys.add(int(c["name"]))
                except (ValueError, TypeError):
                    keys.add(c["name"])
            break
    return keys

invasive_species_keys = _fetch_invasive_facet("degreeOfEstablishment", "invasive")
introduced_keys = _fetch_invasive_facet("establishmentMeans", "introduced")
invasive_species_keys |= introduced_keys
print(f"Invasive/introduced species (GBIF facets, Spain): {len(invasive_species_keys)} speciesKeys")
print(f"Invasive/introduced species (GBIF facets, Spain): {len(introduced_keys)} speciesKeys")

Invasive/introduced species (GBIF facets, Spain): 126 speciesKeys
Invasive/introduced species (GBIF facets, Spain): 126 speciesKeys


In [62]:
# ─── Get total count first (no download) ────────────────────────────────────────
params_count = {
    "geometry": WKT_POLYGON,
    "kingdomKey": KINGDOM_KEY,
    "hasCoordinate": "true",
    "limit": 0,
}
if YEAR is not None:
    params_count["year"] = YEAR
resp = requests.get("https://api.gbif.org/v1/occurrence/search", params=params_count, timeout=TIMEOUT)
resp.raise_for_status()
total_count = resp.json().get("count", 0)
print(f"Records to fetch: {total_count:,}")
if total_count > MAX_RECORDS:
    print(f"⚠️  GBIF API limit is {MAX_RECORDS:,} – only first {MAX_RECORDS:,} will be downloaded.")

Records to fetch: 4,060


In [63]:
records = fetch_occurrences_in_bbox(max_records=MAX_RECORDS_TO_FETCH)
print(f"\nTotal records fetched: {len(records):,}")

Fetching occurrences:   0%|          | 0/10000 [00:00<?, ?rec/s]


Total records fetched: 4,060


## 2 · Aggregate by species and classify invasive status

A species is marked **invasive** if any of:
1. **GBIF record** – occurrence has degreeOfEstablishment/establishmentMeans (rare)
2. **GBIF facet** – speciesKey in degreeOfEstablishment=invasive for this bbox
3. **Known list** – scientific name in KNOWN_INVASIVE_SPECIES (e.g. Cortaderia selloana)

In [64]:
from collections import defaultdict

def aggregate_by_species(records: list[dict]) -> list[dict]:
    agg = defaultdict(lambda: {"total": 0, "invasive": 0, "details": set(), "has_establishment_data": False, "speciesKeys": set()})
    
    for r in records:
        name = r["scientificName"]
        agg[name]["total"] += 1
        if r["is_invasive"]:
            agg[name]["invasive"] += 1
        if r["establishment_detail"] and r["establishment_detail"] != "unknown":
            agg[name]["details"].add(r["establishment_detail"])
            agg[name]["has_establishment_data"] = True
        sk = r.get("speciesKey")
        if sk is not None:
            agg[name]["speciesKeys"].add(int(sk) if isinstance(sk, (int, float)) else sk)
    
    result = []
    for name, v in agg.items():
        from_gbif_record = v["invasive"] > 0
        from_gbif_facet = bool(v["speciesKeys"] & invasive_species_keys)
        name_lower = name.strip().lower()
        name_binom = " ".join(name_lower.split()[:2]) if name_lower.split() else ""
        from_known_list = name_lower in KNOWN_INVASIVE_SPECIES or name_binom in KNOWN_INVASIVE_SPECIES
        is_inv = from_gbif_record or from_gbif_facet or from_known_list
        result.append({
            "scientificName": name,
            "speciesKey": next(iter(v["speciesKeys"]), None) if v["speciesKeys"] else None,
            "total_count": v["total"],
            "invasive_count": v["invasive"],
            "is_invasive": is_inv,
            "invasive_source": "GBIF_record" if from_gbif_record else ("GBIF_facet" if from_gbif_facet else ("known_list" if from_known_list else "")),
            "has_establishment_data": v["has_establishment_data"],
            "establishment_info": "; ".join(sorted(v["details"])) if v["details"] else "",
        })
    
    # Add known invasive species that are not in our records (so we always show them)
    binoms_seen = {" ".join(r["scientificName"].strip().lower().split()[:2]) for r in result if r["scientificName"].strip().split()}
    for known in KNOWN_INVASIVE_SPECIES:
        if known not in binoms_seen:
            result.append({
                "scientificName": known.replace("_", " ").title(),
                "speciesKey": None,
                "total_count": 0,
                "invasive_count": 0,
                "is_invasive": True,
                "invasive_source": "known_list",
                "has_establishment_data": False,
                "establishment_info": "",
            })
    
    return sorted(result, key=lambda x: (-x["total_count"], x["scientificName"]))

In [69]:
species_list = aggregate_by_species(records)
df = pd.DataFrame(species_list)

n_invasive = df["is_invasive"].sum()
n_total = len(df)
print(f"Species with invasive records: {n_invasive} / {n_total}")
print(f"Total occurrence records: {df['total_count'].sum():,}")

# Save for 01_data_collection to read (gbif_deep_learning/species_for_training.csv)
SPECIES_FOR_TRAINING = Path("species_for_training.csv")
df.to_csv(SPECIES_FOR_TRAINING, index=False)
print(f"\nSaved: {SPECIES_FOR_TRAINING.resolve()}")

Species with invasive records: 50 / 789
Total occurrence records: 4,060

Saved: /Users/jakubizewski/Desktop/repos/ie-microsoft-capstone/gbif_deep_learning/species_for_training.csv


In [67]:
df[df['is_invasive']==True]

,scientificName,total_count,invasive_count,is_invasive,invasive_source,has_establishment_data,establishment_info
3,Arctotheca calendula (L.) Levyns,46,0,True,GBIF_facet,False,
11,Cortaderia selloana (Schult. & Schult.f.) Asch...,31,0,True,GBIF_facet,False,
25,Tradescantia fluminensis Vell.,26,0,True,GBIF_facet,False,
26,Laurus nobilis L.,25,0,True,GBIF_facet,False,
48,Oenothera rosea Aiton,16,0,True,GBIF_facet,False,
59,Vinca difformis Pourr.,15,0,True,GBIF_facet,False,
69,Oxalis pes-caprae L.,14,0,True,GBIF_facet,False,
76,"Cymbalaria muralis G.Gaertn., B.Mey. & Scherb.",13,0,True,GBIF_facet,False,
83,Carpobrotus edulis (L.) N.E.Br.,12,0,True,GBIF_facet,False,
105,Tropaeolum majus L.,11,0,True,GBIF_facet,False,


## 3 · Highlight Cortaderia selloana and display results

In [68]:
def highlight_cortaderia(row):
    if TARGET_SPECIES.lower() in (row.get("scientificName") or "").lower():
        return ["background-color: #ffeb3b; font-weight: bold"] * len(row)
    if row.get("is_invasive"):
        return ["background-color: #ffcdd2"] * len(row)  # light red for invasive
    return [""] * len(row)

display(df.style.apply(highlight_cortaderia, axis=1))

,scientificName,total_count,invasive_count,is_invasive,invasive_source,has_establishment_data,establishment_info
0,Linaria triornithophora (L.) Willd.,61,0,False,,False,
1,Struthiopteris spicant (L.) Weiss,54,0,False,,False,
2,Osmunda regalis L.,49,0,False,,False,
3,Arctotheca calendula (L.) Levyns,46,0,True,GBIF_facet,False,
4,Digitalis purpurea L.,41,0,False,,False,
5,Pteridium aquilinum (L.) Kuhn,41,0,False,,False,
6,Erica cinerea L.,39,0,False,,False,
7,Crithmum maritimum L.,38,0,False,,False,
8,Foeniculum vulgare Mill.,34,0,False,,False,
9,Prunella vulgaris L.,34,0,False,,False,


In [ ]:
# Cortaderia selloana specifically
cortaderia = df[df["scientificName"].str.contains(TARGET_SPECIES, case=False, na=False)]
if len(cortaderia) > 0:
    print(f"✓ {TARGET_SPECIES} found in bbox:")
    display(cortaderia)
else:
    print(f"✗ {TARGET_SPECIES} not found in this bbox (no Plantae records with that name).")

In [ ]:
# Invasive species only (sorted by count)
df_invasive = df[df["is_invasive"]].sort_values("invasive_count", ascending=False)
print(f"Invasive species in bbox ({len(df_invasive)} total):")
display(df_invasive)

## 4 · Save to CSV and JSON

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")

export = {
    "bbox": BBOX,
    "kingdom": "Plantae",
    "year": YEAR,
    "total_records": len(records),
    "total_species": len(species_list),
    "invasive_species_count": int(n_invasive),
    "species": species_list,
}
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2, ensure_ascii=False)
print(f"Saved: {OUTPUT_JSON}")